# 02 — Enhancement 1 & 2: chunked ensemble and engineered features

Notebook 01 showed the ceiling: TabPFN v2 takes at most 10,000 in-context rows, so a single context can use only a fraction of NSL-KDD's 125,973 training rows.

This notebook runs the two enhancements against that ceiling, on one fixed test subset so the arms are directly comparable:

| arm | training rows used | features |
|---|---|---|
| **A** single context | 2,500 | 122 |
| **B** chunked ensemble | 20,000 (8 chunks x 2,500) | 122 |
| **C** chunked ensemble + engineered features | 20,000 | 168 |

**Runtime:** about 6-7 minutes.

---

### A note on scale, stated up front

Ensemble cost is `n_chunks x predict_cost(chunk_size, n_test)`. Covering all 125,973 training rows means 51 chunks of 2,500, which costs roughly 2.5s **per test row** on this M1 — a full-coverage run with a usable test sample is several hours, not several minutes.

So this notebook runs a **bounded** ensemble: 8 chunks, 20,000 training rows. That is still 8x what a single context can hold, which is enough to demonstrate the mechanism and to measure whether aggregation helps. It is not the full-dataset result. Section 5 loads the recorded full-scale runs from `reports/` alongside it.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

from tabpfn_nids import config
from tabpfn_nids.data_pipeline import load_and_preprocess_nsl_kdd
from tabpfn_nids.models import ChunkedTabPFNEnsemble, TabPFNWrapper, describe_chunks
from tabpfn_nids.evaluation import compute_metrics, load_results

# Bounded demo configuration. CHUNK_SIZE x MAX_CHUNKS = training rows used.
CHUNK_SIZE = 2_500
MAX_CHUNKS = 8
TEST_ROWS = 400
N_ESTIMATORS = 2   # held equal across every arm, or the comparison would
                   # measure TabPFN ensemble size instead of chunking

config.set_seed(config.SEED)
notebook_started = time.perf_counter()

print(f'device       : {config.resolve_device()}')
print(f'context cap  : {config.MAX_CONTEXT_SAMPLES:,} rows (TabPFN v2 hard limit)')
print(f'ensemble uses: {CHUNK_SIZE * MAX_CHUNKS:,} rows via {MAX_CHUNKS} chunks')

## 1. Load both feature sets

`use_engineered_features` is a single boolean on the preprocessor, so the with/without comparison is a controlled experiment on identical rows and identical splits — the only thing that changes is the feature matrix width.

In [ ]:
X_tr, y_tr, X_te, y_te = load_and_preprocess_nsl_kdd()
X_tr_eng, y_tr_eng, X_te_eng, y_te_eng = load_and_preprocess_nsl_kdd(
    use_engineered_features=True)

# Same rows in the same order, so one index array selects the same records
# from both feature matrices.
assert np.array_equal(y_te, y_te_eng) and np.array_equal(y_tr, y_tr_eng)

print(f'base       : {X_tr.shape[1]} features')
print(f'engineered : {X_tr_eng.shape[1]} features  '
      f'(+{X_tr_eng.shape[1] - X_tr.shape[1]})')

## 2. Fix one test subset

Every arm is scored on exactly these rows. Holding the test set fixed is what makes the deltas in section 4 meaningful.

In [ ]:
test_idx, _ = next(StratifiedShuffleSplit(
    n_splits=1, train_size=TEST_ROWS, random_state=config.SEED).split(X_te, y_te))

X_test, y_test = X_te[test_idx], y_te[test_idx]
X_test_eng = X_te_eng[test_idx]

print(f'test subset: {len(y_test)} rows, attack rate {y_test.mean():.2%}')

## 3. Run the three arms

### Arm A — single context (the baseline mechanism)

One TabPFN, one chunk's worth of training data. This is notebook 01's model at the same scale as a single chunk, so arm B differs from it in exactly one respect: how many chunks get aggregated.

In [ ]:
results, timings = {}, {}

X_a, y_a = X_tr[:CHUNK_SIZE], y_tr[:CHUNK_SIZE]
started = time.perf_counter()

single = TabPFNWrapper(random_state=config.SEED, n_estimators=N_ESTIMATORS,
                       predict_batch_size=1_000)
single.fit(X_a, y_a)
proba_a = single.predict_proba(X_test)
single.free()

timings['A. single context (2,500 rows)'] = time.perf_counter() - started
results['A. single context (2,500 rows)'] = compute_metrics(
    y_test, np.argmax(proba_a, axis=1), proba_a)

print(f"F1 {results['A. single context (2,500 rows)']['f1_score']:.4f}   "
      f"({timings['A. single context (2,500 rows)']:.0f}s)")

### Arm B — chunked ensemble

The training set is partitioned into stratified chunks, each small enough to fit one context. Every chunk predicts on the full test set, and the per-chunk probabilities are aggregated by confidence-weighted voting.

`fit` does no training — it only partitions and stores. All the cost is in `predict`, which runs TabPFN once per chunk.

In [ ]:
started = time.perf_counter()

ensemble = ChunkedTabPFNEnsemble(
    chunk_size=CHUNK_SIZE,
    max_chunks=MAX_CHUNKS,
    aggregation='weighted_vote',
    random_state=config.SEED,
    n_estimators=N_ESTIMATORS,
    stratified=True,
    show_progress=False,
)
ensemble.fit(X_tr, y_tr)
proba_b = ensemble.predict_proba(X_test)

timings['B. chunked ensemble (20,000 rows)'] = time.perf_counter() - started
results['B. chunked ensemble (20,000 rows)'] = compute_metrics(
    y_test, np.argmax(proba_b, axis=1), proba_b)

print(f"F1 {results['B. chunked ensemble (20,000 rows)']['f1_score']:.4f}   "
      f"({timings['B. chunked ensemble (20,000 rows)']:.0f}s)")

#### Did the chunks preserve the class balance?

Stratified chunking is only worth its cost if each chunk really does mirror the population. `max_positive_rate_drift` is the spread between the most and least attack-heavy chunk — it should be near zero.

In [ ]:
described = describe_chunks(ensemble.chunks_)
for key in ('n_chunks', 'min_chunk_size', 'max_chunk_size', 'total_rows',
            'mean_positive_rate', 'max_positive_rate_drift'):
    value = described[key]
    print(f'  {key:<24} {value:.6f}' if isinstance(value, float)
          else f'  {key:<24} {value:,}')

print(f'\npopulation attack rate   {y_tr.mean():.6f}')
print(f'mean chunk confidence    {np.mean(ensemble.chunk_confidences_):.4f}')

### Arm C — chunked ensemble + engineered features

Identical ensemble configuration, identical chunk seeds, identical test rows. The only change is the 46 extra engineered columns (`bytes_ratio`, `total_bytes`, error-rate composites, service/flag combinations).

In [ ]:
started = time.perf_counter()

ensemble_eng = ChunkedTabPFNEnsemble(
    chunk_size=CHUNK_SIZE,
    max_chunks=MAX_CHUNKS,
    aggregation='weighted_vote',
    random_state=config.SEED,
    n_estimators=N_ESTIMATORS,
    stratified=True,
    show_progress=False,
)
ensemble_eng.fit(X_tr_eng, y_tr)
proba_c = ensemble_eng.predict_proba(X_test_eng)

timings['C. ensemble + engineered feats'] = time.perf_counter() - started
results['C. ensemble + engineered feats'] = compute_metrics(
    y_test, np.argmax(proba_c, axis=1), proba_c)

print(f"F1 {results['C. ensemble + engineered feats']['f1_score']:.4f}   "
      f"({timings['C. ensemble + engineered feats']:.0f}s)")

## 4. Comparison table

In [ ]:
score_cols = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']

table = pd.DataFrame(
    {arm: {c: results[arm][c] for c in score_cols} for arm in results}).T
table['runtime_s'] = pd.Series(timings).round(0)
table[score_cols] = table[score_cols].round(4)
table

### The deltas, against the noise floor

A delta only means something if it is larger than the run-to-run variation already present in the baseline. The seed-to-seed F1 standard deviation from the three recorded baseline runs is the threshold to beat.

In [ ]:
baseline_runs = pd.DataFrame(load_results('baseline'))
if len(baseline_runs):
    noise = baseline_runs['f1_score'].astype(float).std()
else:
    noise = float('nan')

f1 = table['f1_score']
arm_a, arm_b, arm_c = table.index

print(f'Seed-to-seed F1 noise floor (3 recorded baseline runs): {noise:.4f} '
      f'({100 * noise:.2f} pp)\n')

for label, delta in [
    ('Enhancement 1  (A -> B, chunking)', f1[arm_b] - f1[arm_a]),
    ('Enhancement 2  (B -> C, features)', f1[arm_c] - f1[arm_b]),
    ('Both           (A -> C)', f1[arm_c] - f1[arm_a]),
]:
    verdict = ('exceeds noise floor' if abs(delta) > noise
               else 'WITHIN noise — not evidence of an effect')
    print(f'{label}: {delta:+.4f} F1  ({100 * delta:+.2f} pp) — {verdict}')

## 5. Against the recorded full-scale runs

Everything above is bounded to fit a 10-minute demo. These are the runs of record in `reports/`, at full context size and larger test samples.

In [ ]:
recorded = pd.DataFrame(load_results('feature_ablation'))
if len(recorded):
    cols = ['arm', 'use_engineered_features', 'n_features', 'n_chunks',
            'context_rows', 'test_rows', 'f1_score', 'roc_auc', 'runtime_seconds']
    print('Recorded feature-engineering ablation (chunked ensemble, seed 42):')
    print(recorded[cols].to_string(index=False))
else:
    print('No feature_ablation CSV in reports/.')

print()
enhanced = load_results('enhanced')
print(f'Recorded full-scale chunked-ensemble runs: {len(enhanced)}'
      + ('  <- not yet run; see scripts/run_enhanced.py' if not enhanced else ''))

## 6. Metrics comparison figure

In [ ]:
from IPython.display import Image, display
from tabpfn_nids.evaluation.plots import plot_metrics_comparison

path = plot_metrics_comparison(
    {arm: {c: results[arm][c] for c in score_cols} for arm in results},
    output_path=config.FIGURES_DIR / 'nb02_enhancement_comparison.png',
    title='NSL-KDD: single context vs chunked ensemble vs +engineered features',
)
display(Image(str(path)))

print(f'Notebook runtime: {time.perf_counter() - notebook_started:.0f}s')

## What to take away

1. **Chunking removes the hard ceiling.** Arm B uses 8x the training data of arm A, which no single TabPFN context can do at any setting. That is the mechanism Enhancement 1 exists to provide.

2. **More data is not automatically more accuracy.** Whether B beats A by more than the noise floor is the empirical question, and the cell in section 4 answers it for this run rather than assuming the answer.

3. **The engineered features have not shown a gain.** The recorded full-scale ablation in section 5 has them *below* the base feature set, by less than the seed noise — which reads as "no measurable effect", not "an improvement". Reporting that honestly is the correct outcome; `reports/ablation_study.md` records it.

4. **Cost scales with chunk count.** Every chunk is a full TabPFN forward pass over the whole test set, so full-dataset coverage is expensive. That trade-off is quantified in notebook 03.